In [ ]:
import requests
import json

### **Pequeño reto para afianzar**
La API de Pokémon tiene un endpoint para obtener la lista de todos los Pokémon (con paginación, igual que JSONPlaceholder). Por ejemplo:

*https://pokeapi.co/api/v2/pokemon?limit=10&offset=0*

- limit es cuántos resultados por página.

- offset es desde qué posición empezar (0 = primeros 10, 10 = siguientes 10, etc.).

Tu misión: Obtén los primeros 20 Pokémon y, para cada uno, imprime su nombre. Luego, para el quinto Pokémon de esa lista, haz una nueva petición a su URL individual (viene en el campo url dentro de cada elemento de results) y muestra su altura y peso.

Esto te preparará para cuando tengas que obtener primero la lista de videos de un canal (que viene paginada) y luego los detalles de cada video.

In [ ]:
lista_pokemon_20 = requests.get("https://pokeapi.co/api/v2/pokemon?limit=20&offset=0")

datos_lista_20 = lista_pokemon_20.json()

print(datos_lista_20)

datos_lista_formateados_20 = json.dumps(datos_lista_20, indent=2)
print(datos_lista_formateados_20)

In [ ]:
count = 0
for i in datos_lista_20["results"]:
    nombre = i["name"]
    count += 1
    print(f"N°{count}: {nombre}")

In [ ]:
url_quinto_pokemon = datos_lista_20["results"][4]["url"]
print(url_quinto_pokemon)

datos_quinto = requests.get(url_quinto_pokemon)

quinto = datos_quinto.json()

#quinto_formateado = json.dumps(quinto, indent=2)
#print(quinto_formateado)

nombre_quinto = quinto["name"]
print(f"¿Quién es ese pokémon?... es {nombre_quinto}")
altura_quinto = quinto["height"]
print(f"La altura del quinto pokemon es: {altura_quinto}")
peso_quinto = quinto["weight"]
print(f"El peso del quinto pokemon es: {peso_quinto}")


### **Reto 1: Explorar un Pokémon aleatorio**

Usa la función random de Python para elegir un número entre 1 y 151 (primera generación).

Construye la URL https://pokeapi.co/api/v2/pokemon/{id}/ con ese número.

Extrae: nombre, peso, altura, tipos (como ya hiciste) y además la primera habilidad oculta (si existe). Las habilidades tienen un campo is_hidden booleano.

Muestra toda la información formateada.

In [ ]:
import random
orden_pokemon = random.randint(1,151)
pokemon_random_json = requests.get(f"https://pokeapi.co/api/v2/pokemon/{orden_pokemon}")

pokemon_random = pokemon_random_json.json()

#pokemon_random_formateado = json.dumps(pokemon_random, indent=2)
#print(pokemon_random_formateado)

nombre_random = pokemon_random["name"]
print(nombre_random)
peso_random = pokemon_random["weight"]
print(peso_random)
altura_random = pokemon_random["height"]
print(altura_random)
lista_tipos = []
for info_tipo in pokemon_random["types"]:
    tipo = info_tipo["type"]["name"]
    lista_tipos.append(tipo)
print(lista_tipos)
lista_hab_escondida = []
for info_hab in pokemon_random["abilities"]:
    if info_hab["is_hidden"] == True:
        habilidad_escondida = info_hab["ability"]["name"]
        lista_hab_escondida.append(habilidad_escondida)
print(lista_hab_escondida)

pokemon_random_formateado = json.dumps(pokemon_random, indent=2)
print(pokemon_random_formateado)


### **Reto 2: Obtener la cadena de evoluciones (esto es más complejo, para profundizar en datos anidados y múltiples peticiones)**

Investiga el endpoint https://pokeapi.co/api/v2/pokemon-species/{id}/ (por ejemplo, para Pikachu, id=25).

De la respuesta, extrae la URL de la cadena evolutiva (evolution_chain.url).

Haz una petición a esa URL para obtener los detalles de las evoluciones.

Extrae los nombres de los Pokémon en la cadena evolutiva (puede ser una sola línea o varias). Por ejemplo, para Pikachu la cadena es Pichu -> Pikachu -> Raichu (dependiendo de condiciones, pero eso lo verás luego).

Muestra los nombres de las evoluciones.

**Indicaciones:**

Para el Reto 2, tendrás que hacer dos peticiones: una para species y otra para evolution-chain.

Los datos en evolution-chain son anidados, tendrás que explorar la estructura JSON para encontrar los nombres.

No te preocupes si no logras todo, el objetivo es practicar la navegación en diccionarios anidados y el manejo de respuestas.

In [ ]:
consulta_evoluciones_json = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/5/")

consulta_evoluciones = consulta_evoluciones_json.json()

#consulta_evoluciones_formateado = json.dumps(consulta_evoluciones, indent=2)
#print(consulta_evoluciones_formateado)

link_evolution_chain = consulta_evoluciones["evolution_chain"]["url"]
evolucion_consulta_json = requests.get(link_evolution_chain)

evolucion_consulta = evolucion_consulta_json.json()

#evolucion_consulta_formateado = json.dumps(evolucion_consulta, indent=2)
#print(evolucion_consulta_formateado)

evolucion_pokemon = evolucion_consulta["chain"]["evolves_to"][0]["species"]["name"]
print(evolucion_pokemon)
evolucion_pokemon_2 = evolucion_consulta["chain"]["species"]["name"]
print(evolucion_pokemon_2)
evolucion_pokemon_3 = evolucion_consulta["chain"]["evolves_to"][0]["evolves_to"][0]["species"]["name"]
print(evolucion_pokemon_3)

### **BFS (Breadth-First Search) – Recorrido por niveles**

In [ ]:
pokemon_a_consultar = random.randint(1,151)
consulta_evoluciones_json = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{pokemon_a_consultar}/")

consulta_evoluciones = consulta_evoluciones_json.json()

#consulta_evoluciones_formateado = json.dumps(consulta_evoluciones, indent=2)
#print(consulta_evoluciones_formateado)

link_evolution_chain = consulta_evoluciones["evolution_chain"]["url"]
evolucion_consulta_json = requests.get(link_evolution_chain)
evolucion_consulta = evolucion_consulta_json.json()

el_chains = evolucion_consulta["chain"]
nodos_por_procesar = [el_chains]
nombres_evoluciones = []

while nodos_por_procesar:
    nodo_actual = nodos_por_procesar.pop(0)
    nombre = nodo_actual["species"]["name"]
    nombres_evoluciones.append(nombre)

    for hijo in nodo_actual["evolves_to"]:
        nodos_por_procesar.append(hijo)

print(nombres_evoluciones)